# 6회차 · 내 문서로 답하는 챗봇 만들기(RAG 샌드박스)

오늘은 **내가 고른 자료**로 검색 기반 생성(RAG)을 직접 만들어 봄. 자료를 넣고, 설정을 바꾸고, 결과가 어떻게 달라지는지 그림과 나란히 놓인 답으로 확인하면서 **내 챗봇**을 완성함.

**전체 흐름부터.** 언어 모델은 학습할 때 본 적 없는 내용(우리 동아리 회칙, 어제 바뀐 공지)을 물으면 모른다고 하거나 그럴듯하게 지어냄. RAG 는 모델을 고치지 않고, 질문과 관련된 문서를 찾아서 **질문 앞에 붙여 주는** 방법임. 책을 펴 놓고 보는 시험과 비슷함.

`내 문서` → 청크로 자름(4단계) → 벡터로 바꿈(5단계) → 질문과 가까운 청크를 찾음(6단계) → 입력에 붙여 모델에게 물음(7단계) → 답

![RAG 전체 그림](img/rag_flow.png)

위쪽 줄은 문서를 넣을 때 **한 번** 해 두는 일(색인), 아래쪽 줄은 질문이 올 때**마다** 하는 일(검색과 생성)임. 질문도 문서와 같은 임베딩 모델로 벡터로 바꿔야 서로 비교할 수 있음. 처음 나오는 말(청크, 임베딩, 벡터DB, top_k 등)은 워크북의 '용어 정리'에 모아 두었음.

그래서 답이 틀렸을 때 원인은 둘 중 하나임. 맞는 청크를 **못 찾았거나**(검색 문제), 찾아서 붙여 줬는데 모델이 **잘못 읽었거나**(생성 문제). 오늘 실험은 대부분 이 둘을 구분하는 연습임.

- 셀은 위에서 아래로 한 번씩 실행함(`Shift + Enter`). 조절판이 나오면 값을 바꾸고 단추를 누름
- 같은 조절판을 몇 번이든 다시 눌러도 됨. 값은 **한 번에 하나**만 바꿔야 무엇 때문에 달라졌는지 알 수 있음
- 내 자료는 서버의 `my/` 폴더에만 남음. 본인과 타인의 실제 개인정보, 외부 반출이 금지된 자료는 넣지 않음
- 끝낼 때 맨 아래 '결과 묶기'를 실행해 zip 파일을 내려받음

| 단계 | 하는 일 | 필수 |
|---|---|---|
| 0 | 시작 : GPU 와 내 프로젝트 상태 | 필수 |
| 1 | 모델 올리기(GPU 계기판) | 필수 |
| 2 | 내 재료 넣기 | 필수 |
| 3 | 내 확인 질문 만들고, 문서 없이 풀어 보기 | 필수 |
| 4 | 자르기 실험(청크) | 선택 |
| 5 | 의미 지도 | 선택 |
| 6 | 찾기 실험(검색) | 선택 |
| 7 | 문서를 붙여 답하기 | 필수 |
| 8 | 설정 실험 : 무엇을 바꾸면 좋아지는가 | 필수 |
| 9 | 모델 비교 | 선택 |
| 10 | 내 챗봇 완성 · 저장 | 필수 |
| 11 | 판단 기록 · 결과 묶기 | 필수 |
| 12 | 벡터DB에 직접 넣고 찾기(코드로 짜 보기) | 더 해 보기 |
| A~H | 임베딩 · 검색을 코드로 더 만져 보기(자율 시간) | 더 해 보기 |

선택 단계(4 · 5 · 6 · 9)는 관심 가는 것을 **둘 이상** 고름. 12단계와 '더 해 보기 A~H'는 필수 단계를 마친 뒤 해 봄. 무엇을 만들지 떠오르지 않으면 맨 아래 '예시 과제'에서 하나를 골라 시작함.

※ 강사가 11/22~23에 개인 서버에서 처음부터 끝까지 직접 돌려 본 노트북임. 숫자는 재료와 서버에 따라 조금씩 달라질수 있음

## 0. 시작
아래 셀을 실행하면 이 서버의 GPU 와 내 프로젝트 상태가 보임. 수업 중에 노트북을 다시 열었을 때도 0단계 셀부터 실행함.

In [ ]:
import sys, pathlib
root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src" / "lab.py").exists())
sys.path.insert(0, str(root / "src"))
import lab
lab.start(6)

## 1. 모델 올리기
언어 모델을 GPU 메모리에 올림. 올린 뒤 계기판에서 **메모리가 얼마나 찼는지**, 여유로 **무엇을 더 올릴 수 있는지** 확인함.

**왜 하나** : 모델은 수십억 개의 숫자(가중치)로 된 파일임. 이 숫자를 GPU 메모리에 올려 두어야 계산이 빠름. 7B급 모델 하나가 약 14GB 를 차지하므로 GPU 메모리 크기가 "어떤 모델을 몇 개 올릴수 있는가"를 정함. 개인 서버의 96GB 는 7B급을 여러 개, 또는 14B 모델까지 올릴 수 있는 크기임.

- 기본 모델 `skt/A.X-4.0-Light`(한국어, 7B급)를 먼저 올림
- 메모리가 넉넉한 서버는 모델을 여러 개 함께 올려 둘 수 있음. 9단계 '모델 비교'에서 씀

In [ ]:
lab.model_panel()

## 2. 내 재료 넣기
챗봇이 답의 근거로 쓸 문서를 넣음. 네 가지 방법을 섞어 써도 됨.
- **직접 쓰기** : 내가 지어낸 설정, 동아리 회칙, 가상의 가게 안내문처럼 모델이 알 수 없는 내용일수록 효과가 잘 보임
- **위키백과** : 관심 주제의 문서 제목을 적으면 가져옴(공개 자료, 출처 자동 기록)
- **예제 재료** : 무엇을 넣을지 정하지 못했으면 준비된 가상 자료 G · L · S 가운데 하나로 시작함
- **파일** : 내 PC 의 `.txt` · `.md` 파일

먼저 '내 주제'를 한 줄로 적어 저장하고, 문서를 **3건 이상** 넣음.

**재료 고르는 요령** : 위키백과 문서만 넣으면 모델이 이미 아는 내용이라 문서를 붙여도 차이가 작게 나옴. 직접 쓴 짧은 글(가격, 날짜, 이름이 들어간 것)을 한 건은 꼭 섞음. 수업에서 가장 차이가 크게 보이는 재료임.

In [ ]:
lab.docs_panel()

## 3. 내 확인 질문
내 문서를 읽어야만 답할 수 있는 질문을 **5건 이상** 적음. 한 줄에 `질문 | 답에 꼭 들어가야 할 단어` 형식임.
기대 단어는 답이 맞았는지 자동으로 판정하는 데 씀(여러 개면 쉼표로 구분).

**왜 하나** : 설정을 바꿀 때마다 "좋아진 것 같음"으로는 판단할 수 없음. 같은 질문 묶음으로 매번 다시 풀어서 점수로 비교해야 함. 오늘 뒤에 나오는 모든 실험이 3단계에서 만든 확인 질문을 씀.

먼저 '둘 다'로 실행해, 문서 없이 풀었을 때와 문서를 붙여 풀었을 때의 점수를 비교함.
- 문서 없이 답한 내용을 읽어 봄. 모른다고 하는가, 그럴듯하게 지어내는가(지어내는 답을 환각이라고 부름)
- 기대 단어는 짧게 씀. `매달 둘째 주 금요일 저녁`보다 `둘째 주 금요일`이 판정에 유리함
- 문서의 표현을 그대로 옮긴 질문과, 일상어로 돌려 말한 질문을 섞어 둠. 6단계에서 차이가 남

In [ ]:
lab.quiz_panel()

## 4. 자르기 실험(선택)
문서는 통째로 넣지 않고 **청크**로 잘라 넣음. 방식과 크기를 바꾸면 경계가 어떻게 달라지는지 봄.

**왜 하나** : 모델이 한 번에 받을 수 있는 글의 길이에는 한도가 있고, 길수록 느리고 비쌈. 그래서 문서를 작은 단위로 잘라 두고 필요한 것만 골라 붙임. 그런데 잘못 자르면 답이 든 문장과 그 문장의 주어가 서로 다른 청크에 들어가 버림.

| 바꿔 볼 것 | 값 | 볼 것 |
|---|---|---|
| 청크 크기 | 150 → 400 → 1000 | 청크 하나만 읽어도 무슨 내용인지 알 수 있는가 |
| 자르는 방식 | 문장 경계 대 글자 수 | 글자 수로 자르면 문장이 중간에 끊기는 곳 |
| 중첩 | 0 → 50 → 150 | 경계에 걸친 문장이 양쪽 청크에 모두 들어가는가 |

![두 가지 자르는 방식(개념도, 문장 길이는 예시)](img/chunk_split.png)

마음에 드는 설정이 나오면 '지금 설정을 내 설정으로 저장'을 켜고 다시 누름.

In [ ]:
lab.chunk_panel()

## 5. 의미 지도(선택)
청크 하나하나를 임베딩 모델이 벡터(숫자 1,024개)로 바꾼 뒤 2차원에 펼친 그림임. 가까이 있는 점은 뜻이 비슷한 청크임.
질문을 적으면 별 표시가 찍히고, 가져온 청크와 선으로 이어짐.

**왜 하나** : 컴퓨터는 글의 뜻을 직접 비교하지 못함. 그래서 글을 숫자 묶음(벡터)으로 바꾸고, 벡터끼리 가까우면 뜻이 비슷하다고 봄. "검색"이 실제로는 질문 벡터와 가까운 청크 벡터를 찾는 계산이라는 것을 그림으로 확인하는 단계임.

가까운 정도는 **코사인 유사도**로 잼. 두 벡터가 가리키는 방향이 같으면 1, 상관없으면 0 근처임. 벡터의 길이를 1로 맞춰 두면 두 벡터의 숫자를 차례로 곱해 더한 값(내적)이 곧 코사인 유사도라서 계산이 빠름.

![임베딩 기초 : 글이 벡터가 되는 과정과 코사인 유사도](img/embed_basics.png)

**임베딩 한 걸음 더**
- 글 → 토큰(글자 묶음)으로 나눔 → 인코더 모델(KURE-v1)이 문맥을 읽음 → 문장 전체를 대표하는 값 하나로 모음 → 길이를 1로 맞춤 → 숫자 1,024개. 위 그림의 아랫줄이 실제 청크 하나를 넣었을때 나온 앞 다섯 숫자임
- 숫자 하나하나에 '뜻'이 붙어 있지는 않음. 1,024개 전체가 가리키는 **방향**이 뜻을 나타냄. 그래서 "몇 번째 숫자가 회비냐" 같은 해석은 안 됨
- 질문과 문서는 꼭 **같은 임베딩 모델**로 바꿔야 비교가 됨. 모델을 바꾸면 색인(청크 벡터)도 처음부터 다시 만듦
- 실제 문장끼리는 무관해도 0.3대가 나옴(KURE-v1 실측 0.34~0.36). 절대값보다 **같은 질문 안에서의 순위**를 봄
- 약한 곳 : 뜻이 반대여도 글자 · 주제가 비슷하면 가깝게 나옴('회비를 올렸다' 대 '내렸다' 0.90). 숫자만 다른 문장도 비슷하게 나옴. 6단계의 키워드 비중이 이걸 보완하는 장치임

※ 처음 해 보면 "유사도 0.6이면 높은 건가?"가 제일 헷갈림. 아래 5-1 셀에서 내 문장 몇 개를 직접 넣어 보면 감이 옴

- 같은 문서의 청크끼리 모여 있는가. 내 질문은 어느 문서 가까이에 떨어지는가
- 질문을 전혀 다른 주제로 바꾸면 별이 어디로 가는지 봄
- 1,024차원을 2차원으로 줄인 그림이라, 그림에서 가장 가까운 점이 순위 1번이 아닐 수 있음. 순위는 아래 표가 정확함

In [ ]:
lab.map_panel()

### 5-1. 임베딩 직접 만져 보기
문장 몇 개를 벡터로 바꾸고, 서로의 코사인 유사도를 표로 봄. `sents` 목록을 내 문장으로 바꿔서 여러 번 돌려 봄.

| 바꿔 볼 것 | 예 | 볼 것 |
|---|---|---|
| 뜻이 반대인 두 문장 | 올렸다 / 내렸다 | 0.9 가까이 나오는가 |
| 같은 뜻, 다른 표현 | 회비 / 가입비 | 반대 뜻보다 낮게 나오는 경우도 있는가 |
| 전혀 다른 주제 | 날씨 문장 | 0.3대에 머무는가 |

In [ ]:
# 5-1. 임베딩 직접 만져 보기 : 문장 몇 개를 벡터로 바꾸고 서로의 코사인 유사도를 표로 봄
sents = ["회비를 올렸다.", "회비를 내렸다.", "동아리 회비는 한 학기에 2만 원이다.",
         "가입비는 얼마인가요?", "내일 서울은 오후부터 비가 온다."]        # 내 문장으로 바꿔 봄
v = lab.embed(sents)                               # 모양 (문장 수, 1024)
print("벡터 모양 :", v.shape)
print("첫 문장 벡터의 앞 5개 숫자 :", v[0][:5].round(3), "· 길이 :", round(float((v[0] ** 2).sum() ** 0.5), 3))
sim = v @ v.T                                      # 길이가 1이므로 내적 = 코사인 유사도
print()
print("     " + "".join(f"  [{j}]" for j in range(len(sents))))
for i, s in enumerate(sents):
    print(f"[{i}]  " + " ".join(f"{x:5.2f}" for x in sim[i]) + f"   {s}")

## 6. 찾기 실험(선택)
검색은 두 점수를 섞어 순위를 정함. **의미 점수**(벡터가 가까운 정도)와 **키워드 점수**(같은 단어가 나온 정도)임.

**왜 하나** : 의미 검색은 "회비 얼마야"를 "가입비 안내" 문서와 이어 줄 만큼 돌려 말한 질문에 강함. 대신 상품 번호나 사람 이름처럼 글자가 정확히 같아야 하는 질문에서는 틀리기도 함. 키워드 검색은 그 반대임. 그래서 둘을 섞어 쓰고, 얼마나 섞을지를 키워드 비중으로 정함.

| 바꿔 볼 것 | 값 | 볼 것 |
|---|---|---|
| 키워드 비중 | 0 → 0.3 → 1 | 답이 든 청크가 몇 순위에 오는가 |
| 질문 종류 | 이름 · 숫자가 든 질문 대 일상어 질문 | 어느 질문이 어느 비중에서 유리한가 |
| 가져올 청크 수 | 2 → 4 → 8 | 답 청크가 목록 안에 들어오는가 |

오늘 노트북은 청크가 수백 개라 질문이 올 때마다 **모든 청크**와 점수를 계산함(전수 비교, numpy). 문서가 수십만 건으로 늘면 이 계산이 느려지므로 벡터DB를 씀. 12단계에서 직접 넣고 찾아 봄.

In [ ]:
lab.search_panel()

## 7. 문서를 붙여 답하기
같은 질문을 문서 없이 한 번, 찾은 청크를 붙여 한 번 물어 나란히 봄. 아래쪽 작은 글씨에서 근거 문서 · 입력 토큰 · 응답 시간을 확인함.

**왜 하나** : 4~6단계는 준비였고, 실제로 답이 바뀌는 곳이 7단계임. '모델에 들어간 입력 보기'를 켜면 모델이 받은 글 전체가 보임. RAG 는 결국 **입력에 문서를 끼워 넣는 일**이고 모델은 바뀌지 않았다는 것을 확인함.

- 입력 토큰이 문서 없이 물었을 때의 몇 배가 됬는지 봄. 붙이는 만큼 느려지고 비용이 늘어남
- 내 문서에 **없는** 내용을 물어봄. 모른다고 답하는가, 붙여 준 문서에서 억지로 답을 만드는가
- 같은 질문에 붙일 청크 수를 1 과 8 로 바꿔 답과 응답 시간을 비교함

In [ ]:
lab.ask_panel()

## 8. 설정 실험
바꿀 설정 하나를 고르고 값 목록을 적으면, 확인 질문 전체를 값마다 다시 풀어 **답 점수 · 검색 적중 · 응답 시간**을 한 그림에 그림.

**왜 하나** : 지금까지는 질문 하나씩 눈으로 봤음. 이제 확인 질문 전체로 재서 "어느 값이 가장 좋은가"를 숫자로 정함. 답 점수가 낮은데 검색 적중이 높으면 생성 문제이고, 검색 적중부터 낮으면 검색 문제임.

| 바꿀 설정 | 값 목록 예 | 예상해 볼 것 |
|---|---|---|
| 붙일 청크 수(top_k) | 1, 2, 4, 8 | 많이 붙일수록 점수가 계속 오르는가, 어디서 멈추는가 |
| 청크 크기 | 150, 400, 1000 | 작게 자르면 검색 적중과 답 점수가 각각 어떻게 변하는가 |
| 키워드 비중 | 0, 0.3, 0.7, 1 | 내 확인 질문에는 의미와 키워드 중 무엇이 더 맞는가 |
| 문서 붙이기 | 끔, 켬 | 3단계 결과를 다시 확인 |

- 실행 전에 어느 값이 가장 좋을지 먼저 예측해 봄. 예측이 틀렸을 때 배우는 것이 가장 많음
- 점수만 보지 않음. 점수가 같으면 더 빠르고 입력이 짧은 설정이 좋은 설정임
- 서로 다른 설정을 **둘 이상** 실험함
- 틀린 질문은 3단계 결과 표의 '검색' 열로 원인을 나눔. '못 찾음'이면 검색을, '찾음'이면 입력과 모델 쪽을 고침

![틀린 질문의 원인 나누기](img/fail_flow.png)

In [ ]:
lab.experiment_panel()

## 9. 모델 비교(선택)
1단계 조절판으로 돌아가 모델을 하나 더 올린 뒤, 같은 질문과 같은 근거 문서를 주고 답과 속도를 비교함.

**왜 하나** : 큰 모델을 쓰는 것이 항상 좋은 선택은 아님. 문서를 잘 찾아서 붙여 주면 작은 모델도 맞히는 경우가 많고, 작은 모델은 몇 배 빠름. 서비스를 만들 때 "이 일에 필요한 가장 작은 모델"을 찾는 것이 비용을 줄이는 방법임.

- 작은 모델(`Qwen3-0.6B`)과 기본 모델의 차이, 다른 계열 모델(`Qwen3-8B`)과의 차이를 봄
- GPU 메모리가 넉넉하면 `Qwen3-14B` 같은 큰 모델도 올릴 수 있음(처음 한 번 내려받음). 계기판의 막대가 얼마나 차는지 같이 봄
- 8단계 조절판에서 바꿀 설정을 '모델'로 고르면 확인 질문 점수로도 비교할 수 있음

In [ ]:
lab.arena_panel()

## 10. 내 챗봇
지금까지 정한 설정으로 대화함. '말투 · 역할' 입력란에 지시를 적으면 답하는 방식이 바뀜.
충분히 써 본 뒤 아래 셀로 **내 솔루션**을 저장함. 7회차는 저장한 솔루션에서 이어 감.

- 옆 사람에게 내 챗봇을 써 보게 함. 내가 생각하지 못한 질문에서 무엇이 틀리는지가 가장 쓸모 있는 정보임
- 말투를 지시문으로 바꾸면 매번 지시문을 붙여야 하고, 긴 대화에서는 지시를 잊기도 함. 7회차에서는 지시문 없이도 그 말투로 답하도록 모델을 학습시킴

In [ ]:
lab.chat()

In [ ]:
lab.save_solution()

## 11. 판단 기록 · 결과 묶기
오늘 실험에서 **무엇을 어떤 값으로 정했고 왜 그렇게 정했는지** 적음. 근거가 되는 숫자를 두 개 이상 넣음(예 : top_k 는 4 로 정함. 8 은 점수가 같은데 응답이 0.4초 느렸음).
기록표와 보고서(`my/report.md`)가 함께 만들어짐. 끝으로 결과 묶기를 실행하고, 왼쪽 파일 창에서 `my_results_*.zip`을 오른쪽 단추 → Download 로 내려받음.

In [ ]:
lab.note_panel()

In [ ]:
lab.pack()

## 12. 벡터DB에 직접 넣고 찾기(더 해 보기)
지금까지는 조절판이 검색을 대신 해 줬음. 12단계에서는 같은 일을 코드 몇 줄로 직접 해 봄. 내 청크를 벡터로 바꾸고 → 벡터DB(벡터 데이터베이스 · Chroma)에 넣고 → 질문 벡터와 가까운 청크를 꺼내고 → 입력을 조립해 모델에 물음.

**왜 하나** : 질문마다 모든 청크와 점수를 계산하는 방식(전수 비교)은 청크가 수백 개일 때는 금방 끝남. 문서가 수십만 건이 되면 질문 하나에 걸리는 시간이 문서 수에 비례해 늘고, 서버를 다시 켤 때마다 벡터를 새로 만들어야 함. 벡터DB는 벡터를 본문 · 제목과 함께 저장해 두고, 근사 최근접 탐색(HNSW)이라는 색인으로 가까운 벡터를 빨리 찾음. 제목 같은 메타데이터로 범위를 좁혀 찾을수도 있음.

![벡터DB가 저장하는 것과 하는 일](img/vectordb_record.png)

- 12-1 : 내 청크를 벡터로 바꿈(`lab.chunks()` · `lab.embed()`). 4단계에서 저장한 자르기 설정을 씀
- 12-2 : Chroma 에 넣음. 서버 메모리에만 두는 방식이라 커널을 다시 시작하면 사라짐
- 12-3 : 질문으로 찾고, 같은 질문을 numpy 전수 비교로 찾은 순위와 나란히 봄
- 12-4 : 찾은 청크로 입력을 조립해 모델에 물음(`lab.llm()`). 7단계 '모델에 들어간 입력 보기'와 같은 모양임
- 12-5 : 메타데이터(문서 제목)로 거른 뒤 찾음

| 바꿔 볼 것 | 값 | 볼 것 |
|---|---|---|
| 질문(12-3의 `q`) | 내 확인 질문 가운데 하나 → 일상어로 돌려 말한 질문 | Chroma 와 전수 비교의 순위가 같은가 |
| `n_results` | 1 → 3 → 8 | 붙이는 청크가 늘면 12-4의 답이 어떻게 바뀌는가 |
| `where` 조건 | 문서 제목 하나 | 범위를 좁히면 순위가 어떻게 바뀌는가 |

- 조절판의 검색은 키워드 점수를 섞으므로(기본 비중 0.3) 6 · 7단계의 순위와 다를 수 있음. 12단계는 의미 점수만 씀

In [ ]:
# 12-1. 내 청크를 벡터로 바꿈
chunks = lab.chunks()                              # 4단계 자르기 설정으로 자른 청크(chunk_id · doc_id · title · text)
vecs = lab.embed([c["text"] for c in chunks])      # 모양 (청크 수, 1024) · 길이를 1로 맞춘 벡터
print(f"청크 {len(chunks)}개 · 벡터 모양 {vecs.shape} · 메모리 {vecs.nbytes / 1024:,.0f}KB")
print("첫 청크 :", chunks[0]["title"], "|", chunks[0]["text"][:60].replace("\n", " "))
print("첫 벡터의 앞 5개 숫자 :", vecs[0][:5].round(3))

In [ ]:
# 12-2. 벡터DB(Chroma)에 넣음
import chromadb
settings = chromadb.Settings(anonymized_telemetry=False)   # 사용 통계를 외부로 보내지 않음
client = chromadb.EphemeralClient(settings=settings)       # 메모리에만 둠. 파일로 남기려면 PersistentClient(path="my/chroma", settings=settings)
try:
    client.delete_collection("my_docs")            # 다시 실행할 때 지난 내용을 지움
except Exception:
    pass
col = client.create_collection("my_docs", metadata={"hnsw:space": "cosine"})   # 코사인 거리로 HNSW 색인을 만듦
col.add(ids=[c["chunk_id"] for c in chunks],
        embeddings=vecs.tolist(),
        documents=[c["text"] for c in chunks],
        metadatas=[{"title": c["title"], "doc_id": c["doc_id"]} for c in chunks])
print("저장한 청크 수 :", col.count())

In [ ]:
# 12-3. 질문으로 찾기 : Chroma(근사 최근접 탐색) 대 numpy 전수 비교
q = lab.quiz()[0]["q"] if lab.quiz() else "여기에 질문을 적음"    # 확인 질문이 있으면 첫 질문을 씀. 직접 바꿔 봄
qv = lab.embed(q)                                  # 질문도 같은 임베딩 모델로 바꿈. 모양 (1, 1024)
res = col.query(query_embeddings=qv.tolist(), n_results=3)
print("질문 :", q)
for rank, (doc, meta, dist) in enumerate(zip(res["documents"][0], res["metadatas"][0], res["distances"][0]), 1):
    print(f"  Chroma {rank}위  유사도 {1 - dist:.3f}  {meta['title'][:16]} | {doc[:44]}".replace("\n", " "))
scores = vecs @ qv[0]                              # 전수 비교 : 모든 청크와 내적(= 코사인 유사도)
for rank, i in enumerate(scores.argsort()[::-1][:3], 1):
    print(f"  전수   {rank}위  유사도 {scores[i]:.3f}  {chunks[i]['title'][:16]} | {chunks[i]['text'][:44]}".replace("\n", " "))

In [ ]:
# 12-4. 찾은 청크로 입력을 조립해 모델에 물음
context = "\n\n".join(f"<{m['title']}>\n{d}" for d, m in zip(res["documents"][0], res["metadatas"][0]))
prompt = f"[참고 문서]\n{context}\n\n[질문]\n{q}"
print("시스템 규칙 :", lab.RAG_RULE)
print("입력 :", prompt[:400], "…" if len(prompt) > 400 else "")
print("\n답 :", lab.llm(prompt, system=lab.RAG_RULE))

In [ ]:
# 12-5. 메타데이터로 거른 뒤 찾기 : 문서 제목 하나 안에서만 찾음
title = res["metadatas"][0][0]["title"]            # 12-3의 1위 청크가 나온 문서
res2 = col.query(query_embeddings=qv.tolist(), n_results=3, where={"title": title})
print(f"'{title}' 안에서만 찾은 청크 {len(res2['ids'][0])}개")
for doc, dist in zip(res2["documents"][0], res2["distances"][0]):
    print(f"  유사도 {1 - dist:.3f} | {doc[:60]}".replace("\n", " "))

---
## 더 해 보기 A~H(자율 시간)
필수 단계를 마쳤으면 아래에서 마음에 드는 것부터 골라 해 봄. 순서는 상관없음. 셀마다 위쪽 몇 줄(질문, 문장 목록)만 내 재료에 맞게 고치면 됨. 워크북 '더 해 보기' 절에 같은 목록과 적을 것이 정리돼 있음.

| 번호 | 할 일 | 걸리는 시간 | 볼 것 |
|---|---|---|---|
| A | 문장 쌍 유사도 실험 | 5분 | 임베딩이 둔한 차이(반대 뜻 · 숫자)와 민감한 차이 |
| B | 유사도 표를 그림으로 | 5분 | 비슷한 문장끼리 진하게 모이는가 |
| C | 같은 질문을 세 가지로 바꿔 묻기 | 10분 | 표현을 바꾸면 순위가 얼마나 달라지는가 |
| D | 키워드 비중 0 → 1 | 5분 | 1위 청크가 바뀌는 지점 |
| E | 직접 짠 검색 대 조절판 검색 | 5분 | 의미 점수만 쓰면 순위가 같은가 |
| F | 설정 실험을 코드로 | 10분 | 청크 크기 · top_k를 바꿨을 때 점수와 입력 토큰 |
| G | 문서에 없는 질문 | 5분 | 모른다고 답하는가, 지어내는가 |
| H | 스스로 점검 다섯 문제 | 5분 | 접힌 답과 내 답 비교 |

※ 셀을 돌리다 오류가 나면 대부분 앞 단계(모델 올리기, 재료 넣기, 확인 질문)를 안 한 경우임. 0단계부터 다시 실행하면 됨

### A. 문장 쌍 유사도 실험
반대 뜻, 숫자만 다른 문장, 같은 뜻의 다른 표현, 다른 언어, 띄어쓰기 차이를 한 줄씩 비교함. 직접 쌍을 두세 개 더 넣어 봄.

In [ ]:
# A. 문장 쌍 유사도 실험 : 어떤 차이에 둔하고 어떤 차이에 민감한가
pairs = [("회비를 올렸다.", "회비를 내렸다."),                          # 반대 뜻
         ("회비는 2만 원이다.", "회비는 3만 원이다."),                    # 숫자만 다름
         ("회비는 얼마인가요?", "가입비가 얼마예요?"),                    # 같은 뜻, 다른 표현
         ("회비는 얼마인가요?", "How much is the membership fee?"),       # 같은 뜻, 다른 언어
         ("회비는 얼마인가요?", "회비는얼마인가요"),                        # 띄어쓰기 · 문장부호
         ("회비는 얼마인가요?", "김치찌개는 돼지고기를 넣으면 맛있다.")]     # 무관
for a, b in pairs:
    va, vb = lab.embed([a, b])
    print(f"{float(va @ vb):.3f}   {a}  |  {b}")

### B. 유사도 표를 그림으로
5-1의 문장 목록(`sents`)으로 그림을 그림. 5-1에서 문장을 바꾼 뒤 이 셀을 다시 돌리면 그림도 바뀜.

In [ ]:
# B. 유사도 표를 그림으로 : 진할수록 가까움
try:
    sents
except NameError:
    sents = ["회비를 올렸다.", "회비를 내렸다.", "동아리 회비는 한 학기에 2만 원이다.", "가입비는 얼마인가요?", "내일 서울은 오후부터 비가 온다."]
plt = lab._plt()                                   # 한글 글꼴을 맞춘 matplotlib
v = lab.embed(sents)
sim = v @ v.T
fig, ax = plt.subplots(figsize=(6.4, 4.4))
im = ax.imshow(sim, vmin=0, vmax=1, cmap="Blues")
ax.set_xticks(range(len(sents)))
ax.set_xticklabels([str(i) for i in range(len(sents))])
ax.set_yticks(range(len(sents)))
ax.set_yticklabels([f"{i} {s[:14]}" for i, s in enumerate(sents)])
for i in range(len(sents)):
    for j in range(len(sents)):
        ax.text(j, i, f"{sim[i, j]:.2f}", ha="center", va="center", fontsize=9, color="white" if sim[i, j] > 0.7 else "black")
fig.colorbar(im, ax=ax, shrink=0.8)
ax.set_title("문장끼리의 코사인 유사도")
plt.show()

### C. 같은 질문을 세 가지로 바꿔 묻기
같은 뜻을 정식 문장, 일상어, 낱말 두 개로 바꿔 물어 봄. 의미 검색만(kw=0)과 섞기(kw=0.3)를 나란히 봄. `qs`를 내 재료에 맞는 질문으로 고침.

In [ ]:
# C. 같은 질문을 세 가지로 : 표현을 바꾸면 순위가 얼마나 달라지는가
qs = ["회비는 얼마인가요?", "한 학기에 돈을 얼마나 내요?", "회비 금액"]      # 내 재료에 맞게 바꿈
for q in qs:
    for kw in (0, 0.3):
        hits = lab.search(q, k=3, kw=kw)
        print(f"kw={kw:<3}  {q[:18]:<18} → " + " | ".join(f"{h['rank']}. {h['title'][:12]}" for h in hits))
    print()

### D. 키워드 비중 0 → 1
확인 질문 하나로 키워드 비중을 0에서 1까지 바꾸며 1위 청크가 언제 바뀌는지 봄. 이름 · 숫자가 든 질문과 일상어 질문을 번갈아 넣어 보면 차이가 잘 보임.

In [ ]:
# D. 키워드 비중을 바꾸며 1위 청크를 봄(의미 = 코사인 유사도, 키워드 = BM25 점수)
q = lab.quiz()[0]["q"] if lab.quiz() else "여기에 질문을 적음"
print("질문 :", q)
for kw in [0, 0.2, 0.4, 0.6, 0.8, 1.0]:
    h = lab.search(q, k=1, kw=kw)[0]
    print(f"kw={kw:.1f}  의미 {h['vec']:.3f}  키워드 {h['kwd']:6.2f}  → {h['title'][:14]} | {h['text'][:40]}".replace("
", " "))

### E. 직접 짠 검색 대 조절판 검색
`lab.chunks()`와 `lab.embed()`로 전수 비교를 직접 짜고, 조절판이 쓰는 `lab.search()`와 순위를 비교함. 키워드 비중을 0으로 두면 둘 다 코사인 유사도만 쓰므로 같아야 함.

In [ ]:
# E. 직접 짠 검색(numpy 전수 비교) 대 lab.search(kw=0)
import numpy as np
q = lab.quiz()[0]["q"] if lab.quiz() else "여기에 질문을 적음"
chunks = lab.chunks()
vecs = lab.embed([c["text"] for c in chunks])
mine = [chunks[i]["chunk_id"] for i in np.argsort(-(vecs @ lab.embed(q)[0]))[:5]]
theirs = [h["chunk_id"] for h in lab.search(q, k=5, kw=0)]
print("직접   :", mine)
print("조절판 :", theirs)
print("순위가 같음" if mine == theirs else "순위가 다름 → kw를 0.3으로 바꾸면 어떻게 되는지도 봄")

### F. 설정 실험을 코드로
8단계 조절판과 같은 함수(`lab.experiment`)를 코드로 부름. 값 목록만 바꿔 여러 번 돌려 봄. 바꿀 수 있는 설정은 `k` · `kw` · `size` · `overlap` · `method` · `rag` · `model`.

In [ ]:
# F. 청크 크기를 바꿔 가며 확인 질문을 풂(값 하나에 몇 초씩 걸림)
lab.experiment("size", [200, 400, 800])

In [ ]:
# F-2. 붙일 청크 수(top_k)를 바꿔 봄
lab.experiment("k", [1, 2, 4, 8])

### G. 문서에 없는 질문
내 재료에 **없는** 내용을 세 가지 물어 봄. 지시문(`lab.RAG_RULE`)대로 모른다고 하는지, 그럴듯하게 지어내는지 봄. 지어낸다면 판단 기록에 한 줄 남김.

In [ ]:
# G. 문서에 없는 질문 세 개(내 재료에 맞게 바꿈)
for q in ["동아리를 처음 만든 해는 언제인가요?", "다음 달 회비 인상 계획이 있나요?", "작년 겨울 MT 장소는 어디였나요?"]:
    lab.ask(q, k=2)

### H. 스스로 점검
답을 먼저 한두 줄로 적어 보고 펼쳐서 비교함.

<details><summary>1. 질문은 KURE-v1, 문서는 다른 임베딩 모델로 바꾸면 어떻게 되는가</summary>

두 벡터가 서로 다른 모델의 숫자라 유사도가 의미 없는 값이 됨. 검색이 관계없는 청크를 가져옴. 모델을 바꾸면 문서 쪽 색인도 다시 만들어야 함
</details>

<details><summary>2. 무관한 문장끼리 유사도가 0.35 나왔다. 이상한가</summary>

이상하지 않음. 문장 임베딩은 대부분 0.3~0.9 사이에 모임. 절대값보다 같은 질문 안에서의 순위를 봄
</details>

<details><summary>3. '회비를 올렸다'와 '회비를 내렸다'가 0.90이다. 검색에서 무엇이 문제가 되고, 어떻게 보완하는가</summary>

인상 공지를 물었는데 인하 공지가 위에 올 수 있음. 키워드 점수를 섞거나(6단계), 재순위화 모델로 순서를 고치거나, 날짜 · 버전 같은 메타데이터로 거름
</details>

<details><summary>4. 청크를 아주 크게(문서 하나 통째로) 자르면 무엇이 좋아지고 무엇이 나빠지는가</summary>

문맥은 남지만 관련 없는 내용이 함께 붙고 입력 토큰과 지연이 늘어남. 임베딩 모델의 입력 한도를 넘으면 뒤가 잘려 벡터에 반영되지 않음
</details>

<details><summary>5. 내 챗봇의 청크가 300개다. 벡터DB가 꼭 필요한가</summary>

꼭 필요하지는 않음. 코어 하나로도 1천 개 전수 비교가 0.045ms(11/23 실측). 서버를 다시 켜도 남겨야 하거나, 메타데이터로 거르거나, 여러 사람이 함께 쓸 때 필요해짐
</details>

---
### 예시 과제(무엇을 만들지 정하지 못했을 때 하나를 고름)
1. **우리 동아리(또는 가게) 안내 챗봇** : 회칙 · 회비 · 모임 시간 · 대여 규정을 직접 써서 넣음(500자면 충분함). 확인 질문 5건을 모두 맞히는 가장 작은 top_k 를 찾음
2. **내가 지어낸 세계관 안내자** : 게임이나 소설 설정(인물 · 지명 · 규칙)을 지어내 넣고, 위키백과 문서 한두 건을 함께 넣음. 지어낸 설정과 실제 지식을 섞어 물었을 때 어느 문서를 근거로 답하는지 봄
3. **과목 요약 노트 챗봇** : 이번 학기 과목의 내 필기(공개해도 되는 내용만)를 넣고 시험 예상 질문을 확인 질문으로 적음. 일상어로 돌려 물어도 찾는지 6단계에서 확인함

### 도전 과제
- **틀린 문서 넣기** : 2단계에서 일부러 사실과 다른 문서를 넣고 답이 어떻게 바뀌는지 봄. 모델은 붙여 준 문서를 믿음. 문서 관리가 왜 중요한지 한 줄로 적음
- **기준일이 다른 문서 두 건** : "회비 2만 원(3월 기준)"과 "회비 3만 원(9월 기준)"을 넣고 어느 것으로 답하는지, 지시문에 "가장 최근 기준으로 답한다"를 넣으면 달라지는지 봄
- **가장 싼 설정 찾기** : 답 점수 1.0 을 유지하면서 입력 토큰이 가장 적은 설정(청크 크기와 top_k 의 조합)을 찾음
- **검색 실패 일부러 만들기** : 문서에 답은 있는데 검색이 못 찾는 질문을 하나 만들어 봄. 어떻게 물으면 못 찾는가
- **벡터DB를 파일로 남기기** : 12-2 셀의 `EphemeralClient(settings=settings)` 를 `PersistentClient(path="my/chroma", settings=settings)` 로 바꿔 실행하고, 커널을 다시 시작한 뒤 `chromadb.PersistentClient(path="my/chroma").get_collection("my_docs").count()` 가 그대로인지 봄
- 조절판 없이 직접 부르기 : `lab.ask("질문", k=2)`, `lab.experiment("size", [200, 400, 800])`, `lab.board()`
- 5회차의 도메인 과제(JSON 답 채점)를 이 서버에서 재려면 : `lab.sh("python src/my_start.py --domain G")` 뒤 `lab.sh("python src/benchmark.py --name baseline --limit 10")`